# Comparação entre Política Aleatória e Política Básica no Blackjack

## Objetivo
Este notebook compara duas estratégias de decisão para o jogo Blackjack:

- **Política aleatória**, que escolhe ações sem critério;
- **Política básica**, que segue uma regra simples baseada na soma atual da mão do jogador.

O objetivo é criar uma linha de base para avaliar o desempenho de estratégias simples antes de introduzir o agente treinado com Q-Learning.

## Papel desta etapa no projeto

Esta etapa funciona como uma referência inicial para o projeto.

Antes de avaliar um agente treinado com aprendizado por reforço, é importante medir o desempenho de estratégias simples. Assim, os resultados obtidos aqui servirão como base de comparação para as próximas fases.

## Reutilização do ambiente

As funções principais do ambiente Blackjack não são redefinidas neste notebook.

A lógica do jogo foi centralizada no arquivo `blackjack_env.py`, que contém funções como:

- `comprar_carta()`
- `valor_mao()`
- `tem_as_utilizavel()`
- `estado_jogo()`
- `partida_blackjack()`

Essa organização evita duplicação de código e torna o projeto mais limpo, reutilizável e fácil de manter.

## Importações e configuração inicial

Nesta seção são importadas as bibliotecas utilizadas e definida uma semente aleatória para tornar os experimentos reproduzíveis.

In [ ]:
import pandas as pd
import random

from utils_io import salvar_df

random.seed(42)

In [ ]:
from blackjack_env import comprar_carta, valor_mao, tem_as_utilizavel, estado_jogo, partida_blackjack

## Reutilização do ambiente

As funções principais do ambiente Blackjack não são redefinidas neste notebook.

A lógica do jogo foi centralizada no arquivo `blackjack_env.py`, que contém funções como:

- `comprar_carta()`
- `valor_mao()`
- `tem_as_utilizavel()`
- `estado_jogo()`
- `partida_blackjack()`

Essa organização evita duplicação de código e torna o projeto mais limpo, reutilizável e fácil de manter.

In [3]:
def politica_aleatoria(estado):
    return random.choice(["hit", "stick"])

def politica_basica(estado):
    soma_jogador, carta_dealer, as_utilizavel = estado

    if soma_jogador < 17:
        return "hit"
    return "stick"

## Validação inicial da simulação

Antes da execução em larga escala, é útil testar uma partida individual para verificar se:

- a estrutura retornada pela função está correta;
- os campos esperados estão sendo gerados;
- a política está interagindo corretamente com o ambiente.

In [4]:
def partida_blackjack(policy, id_episodio=0):
    mao_jogador = [comprar_carta(), comprar_carta()]
    mao_dealer = [comprar_carta(), comprar_carta()]

    historico = []
    passo = 0

    while True:
        estado = estado_jogo(mao_jogador, mao_dealer[0])
        soma_atual = valor_mao(mao_jogador)

        if soma_atual >= 21:
            break

        acao = policy(estado)

        historico.append({
            "id_episodio": id_episodio,
            "passo": passo,
            "soma_jogador": estado[0],
            "carta_dealer": estado[1],
            "as_utilizavel": estado[2],
            "acao": acao
        })

        if acao == "hit":
            mao_jogador.append(comprar_carta())

            if valor_mao(mao_jogador) > 21:
                return {
                    "id_episodio": id_episodio,
                    "resultado": "derrota",
                    "recompensa": -1,
                    "total_jogador": valor_mao(mao_jogador),
                    "total_dealer": valor_mao(mao_dealer),
                    "dealer_carta_visivel": mao_dealer[0],
                    "qtd_passos": passo + 1,
                    "mao_jogador": str(mao_jogador),
                    "mao_dealer": str(mao_dealer),
                    "historico": historico
                }
        else:
            break

        passo += 1

    while valor_mao(mao_dealer) < 17:
        mao_dealer.append(comprar_carta())

    total_jogador = valor_mao(mao_jogador)
    total_dealer = valor_mao(mao_dealer)

    if total_dealer > 21 or total_jogador > total_dealer:
        resultado = "vitoria"
        recompensa = 1
    elif total_jogador < total_dealer:
        resultado = "derrota"
        recompensa = -1
    else:
        resultado = "empate"
        recompensa = 0

    return {
        "id_episodio": id_episodio,
        "resultado": resultado,
        "recompensa": recompensa,
        "total_jogador": total_jogador,
        "total_dealer": total_dealer,
        "dealer_carta_visivel": mao_dealer[0],
        "qtd_passos": passo + 1,
        "mao_jogador": str(mao_jogador),
        "mao_dealer": str(mao_dealer),
        "historico": historico
    }

In [5]:
teste = partida_blackjack(politica_basica, id_episodio=1)
teste

{'id_episodio': 1,
 'resultado': 'vitoria',
 'recompensa': 1,
 'total_jogador': 19,
 'total_dealer': 18,
 'dealer_carta_visivel': 6,
 'qtd_passos': 1,
 'mao_jogador': '[10, 9]',
 'mao_dealer': '[6, 2, 10]',
 'historico': [{'id_episodio': 1,
   'passo': 0,
   'soma_jogador': 19,
   'carta_dealer': 6,
   'as_utilizavel': 0,
   'acao': 'stick'}]}

## Simulação em múltiplos episódios

Para comparar as políticas de forma robusta, são simulados muitos episódios para cada estratégia.

A simulação gera duas bases:
- uma com os resultados finais de cada episódio;
- outra com o histórico das decisões tomadas ao longo das partidas.

Essas bases permitem comparar desempenho e comportamento das políticas.

In [6]:
def simular_partidas(policy, nome_politica, n_episodios=100000):
    episodios = []
    decisoes = []

    for episodio in range(n_episodios):
        resultado = partida_blackjack(policy, id_episodio=episodio)

        episodios.append({
            "id_episodio": resultado["id_episodio"],
            "politica": nome_politica,
            "resultado": resultado["resultado"],
            "recompensa": resultado["recompensa"],
            "total_jogador": resultado["total_jogador"],
            "total_dealer": resultado["total_dealer"],
            "dealer_carta_visivel": resultado["dealer_carta_visivel"],
            "qtd_passos": resultado["qtd_passos"],
            "mao_jogador": resultado["mao_jogador"],
            "mao_dealer": resultado["mao_dealer"]
        })

        for d in resultado["historico"]:
            d["politica"] = nome_politica
            decisoes.append(d)

    df_episodios = pd.DataFrame(episodios)
    df_decisoes = pd.DataFrame(decisoes)

    return df_episodios, df_decisoes

## Execução dos experimentos

Cada política é executada em **100000 episódios**.

Esse volume foi escolhido para reduzir a variação aleatória dos resultados e tornar a comparação mais confiável.

In [7]:
df_epi_aleatoria, df_dec_aleatoria = simular_partidas(
    policy=politica_aleatoria,
    nome_politica="aleatoria",
    n_episodios=100000
)

df_epi_basica, df_dec_basica = simular_partidas(
    policy=politica_basica,
    nome_politica="basica",
    n_episodios=100000
)

## Consolidação dos resultados

Após a simulação das duas políticas, os resultados são reunidos em bases consolidadas para facilitar a análise comparativa entre as estratégias.

In [8]:
df_episodios = pd.concat([df_epi_aleatoria, df_epi_basica], ignore_index=True)
df_decisoes = pd.concat([df_dec_aleatoria, df_dec_basica], ignore_index=True)

df_episodios.head()

,id_episodio,politica,resultado,recompensa,total_jogador,total_dealer,dealer_carta_visivel,qtd_passos,mao_jogador,mao_dealer
0,0,aleatoria,vitoria,1,12,26,2,1,"[10, 2]","[2, 10, 4, 10]"
1,1,aleatoria,derrota,-1,12,19,8,1,"[2, 10]","[8, 8, 3]"
2,2,aleatoria,derrota,-1,24,12,9,1,"[10, 10, 4]","[9, 3]"
3,3,aleatoria,derrota,-1,19,20,10,3,"[5, 4, 8, 2]","[10, 10]"
4,4,aleatoria,vitoria,1,20,23,10,1,"[10, 10]","[10, 5, 8]"


## Distribuição dos resultados por política

A primeira análise observa quantos episódios terminaram em vitória, derrota e empate para cada política.

Essa comparação permite identificar rapidamente qual estratégia apresenta melhor comportamento geral.

In [9]:
comparacao = (
    df_episodios
    .groupby(["politica", "resultado"])
    .size()
    .reset_index(name="qtd")
)

comparacao

,politica,resultado,qtd
0,aleatoria,derrota,3173
1,aleatoria,empate,219
2,aleatoria,vitoria,1608
3,basica,derrota,2421
4,basica,empate,560
5,basica,vitoria,2019


## Métricas consolidadas

Além da distribuição dos resultados, são calculadas métricas agregadas por política, como:

- número total de episódios;
- recompensa média;
- total médio da mão do jogador;
- número médio de passos por episódio.

Esses indicadores ajudam a entender o desempenho médio e o estilo de decisão de cada estratégia.

In [10]:
metricas = (
    df_episodios
    .groupby("politica")
    .agg(
        episodios=("id_episodio", "count"),
        recompensa_media=("recompensa", "mean"),
        total_medio_jogador=("total_jogador", "mean"),
        passos_medios=("qtd_passos", "mean")
    )
    .reset_index()
)

metricas

,politica,episodios,recompensa_media,total_medio_jogador,passos_medios
0,aleatoria,5000,-0.3130,18.3550,1.3416
1,basica,5000,-0.0804,20.3454,1.6238


## Taxa de vitória

A taxa de vitória resume a proporção de partidas vencidas por cada política e funciona como uma das métricas mais intuitivas desta etapa.

In [11]:
taxa_vitoria = (
    df_episodios.assign(vitoria=lambda x: (x["resultado"] == "vitoria").astype(int))
    .groupby("politica")["vitoria"]
    .mean()
    .reset_index(name="taxa_vitoria")
)

taxa_vitoria

,politica,taxa_vitoria
0,aleatoria,0.3216
1,basica,0.4038


## Persistência dos resultados

Os dados gerados nesta etapa são salvos em arquivos CSV para reutilização nas fases seguintes do projeto, especialmente na avaliação final e na análise exploratória.

In [12]:
salvar_df(df_episodios, "blackjack_episodios_etapa_basica", pasta="dados")
salvar_df(df_decisoes, "blackjack_decisoes_etapa_basica", pasta="dados")
salvar_df(metricas, "blackjack_metricas_etapa_basica", pasta="resultados")
salvar_df(taxa_vitoria, "blackjack_taxa_vitoria_etapa_basica", pasta="resultados")

Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_episodios_etapa_basica.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_decisoes_etapa_basica.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_metricas_etapa_basica.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_taxa_vitoria_etapa_basica.csv
